In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx']}

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_data_path = Path(data_path) / 'Historical' / index_data_files[0]

# Read raw data
index_data_raw = pd.read_excel(index_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['Country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    # make first row columns names
    index_data_raw.columns = index_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

In [9]:
arab_countries = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                  'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']
# Index data minor reformatting and filtering
index_data = index_data_raw.copy()
index_data = index_data.rename(columns={'Unnamed: 0':'KeyID','Unnamed: 1':'Series name','Unnamed: 2':'Year',
                                        'year':'Year'})
index_data = index_data.iloc[1:]

# Long format, arab countries
index_data_long = index_data.melt(id_vars=['Series name','KeyID','Year'], var_name='ISO', value_name='data').copy()       # convert wide to long
index_data_long = index_data_long[index_data_long['ISO'].isin(arab_countries)] 

index_data_long['data'] = pd.to_numeric(index_data_long['data'], errors='coerce')
index_data_long = index_data_long.merge(country_iso[['ISO','Country']],how='left')
index_data_long= index_data_long[['KeyID','Series name', 'Country', 'ISO', 'Year', 'data']]

In [20]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
def write_dataframes_to_excel(
    dataframes: dict,
    output_path: str = "output.xlsx",
    freeze_header: bool = True,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,#"4472C4",   # Excel-blue; set None for no fill
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    add_autofilter: bool = True,
    index: bool = False,
):
    """
    Write a dict of DataFrames to one Excel file, one sheet per DataFrame.
 
    Parameters
    ----------
    dataframes : dict
        Mapping of {sheet_name: dataframe}. Sheet names are what you control
        naming for — just use whatever keys you want.
        Example: {"Sales": df_sales, "Inventory": df_inv}
    output_path : str
        Path to save the .xlsx file.
    freeze_header : bool
        If True, freezes the header row so it stays visible when scrolling.
    autofit_columns : bool
        If True, resizes each column to fit its longest value.
    max_col_width : int
        Cap on auto-fit column width (prevents huge columns from long text).
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font color.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        If True, adds thin borders around all cells.
    add_autofilter : bool
        If True, adds filter dropdowns to the header row.
    index : bool
        Whether to write the DataFrame index as a column.
    """
 
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dataframes.items():
            # Excel sheet names max 31 chars, no : \ / ? * [ ]
            safe_name = _sanitize_sheet_name(sheet_name)
            df.to_excel(writer, sheet_name=safe_name, index=index)
 
            worksheet = writer.sheets[safe_name]
            _format_worksheet(
                worksheet,
                df,
                freeze_header=freeze_header,
                autofit_columns=autofit_columns,
                max_col_width=max_col_width,
                header_bg_color=header_bg_color,
                header_font_color=header_font_color,
                header_bold=header_bold,
                add_borders=add_borders,
                add_autofilter=add_autofilter,
                index=index,
            )
 
    print(f"Saved: {output_path}")
 
 
def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]
 
 
def _format_worksheet(
    ws,
    df,
    freeze_header,
    autofit_columns,
    max_col_width,
    header_bg_color,
    header_font_color,
    header_bold,
    add_borders,
    add_autofilter,
    index,
):
    n_cols = df.shape[1] + (1 if index else 0)
    n_rows = df.shape[0] + 1  # +1 for header row
 
    # --- Header styling ---
    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        if header_fill:
            cell.fill = header_fill
        cell.alignment = header_align
 
    # Slightly taller header row to fit wrapped text
    ws.row_dimensions[1].height = 22
 
    # --- Optional thin borders on all used cells ---
    if add_borders:
        thin = Side(style="thin", color="B7B7B7")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=1, max_row=n_rows, min_col=1, max_col=n_cols):
            for cell in row:
                cell.border = border
 
    # --- Freeze header row (and index column if present) ---
    if freeze_header:
        ws.freeze_panes = "B2" if index else "A2"
 
    # --- Autofilter on header row ---
    if add_autofilter:
        ws.auto_filter.ref = ws.dimensions
 
    # --- Autofit column widths based on longest value ---
    if autofit_columns:
        for col_idx in range(1, n_cols + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

In [21]:
def structural_metrics(df, group_cols=['KeyID','Series name','Country', 'ISO'], year_col='Year', value_col='data', min_run=3):
    
    # max year per KeyID (used as the reference/anchor for last_5 / last_10 windows)
    keyid_max_year = df.groupby('KeyID')[year_col].max().to_dict()
    
    results = []
    for keys, g in df.groupby(group_cols):
        g = g.sort_values(year_col)
        years = g[year_col].dropna().astype(int)
        actual_years = set(years)
        
        # years where a row exists AND value is not null
        actual_years_nonnull = set(g.loc[g[value_col].notna(), year_col].dropna().astype(int))
        
        expected_range = set(range(years.min(), years.max() + 1)) if len(years) else set()
        missing_years = sorted(expected_range - actual_years)
        
        # longest consecutive gap (based on missing rows, not nulls)
        gaps = []
        if missing_years:
            gap_run = [missing_years[0]]
            for y in missing_years[1:]:
                if y == gap_run[-1] + 1:
                    gap_run.append(y)
                else:
                    gaps.append(gap_run)
                    gap_run = [y]
            gaps.append(gap_run)
        longest_gap = max((len(g) for g in gaps), default=0)
        
        # reference year is based on this row's KeyID
        this_keyid = keys[group_cols.index('KeyID')]
        reference_year = int(keyid_max_year[this_keyid])
        last_5_range = set(range(reference_year - 4, reference_year + 1))
        last_10_range = set(range(reference_year - 9, reference_year + 1))
        
        # last 5 / 10 years availability -- now based on non-null values, not just row presence
        last_5_available = len(actual_years_nonnull & last_5_range)
        last_10_available = len(actual_years_nonnull & last_10_range)

        # Min max non-null years
        min_year_nonnull = min(actual_years_nonnull) if actual_years_nonnull else None
        max_year_nonnull = max(actual_years_nonnull) if actual_years_nonnull else None
        
        # consecutive repeated values (3+ years)
        g_valid = g.dropna(subset=[year_col, value_col])
        run_years = g_valid[year_col].astype(int).tolist()
        run_values = g_valid[value_col].tolist()
        
        ranges = []
        run_start_idx = 0
        for i in range(1, len(run_years) + 1):
            same_run = (
                i < len(run_years)
                and run_values[i] == run_values[i - 1]
                and run_years[i] == run_years[i - 1] + 1
            )
            if not same_run:
                run_length = i - run_start_idx
                if run_length >= min_run:
                    start_year = run_years[run_start_idx]
                    end_year = run_years[i - 1]
                    ranges.append(f"{start_year}-{end_year}")
                run_start_idx = i
        
        results.append({
            **dict(zip(group_cols, keys)),
            'n_records': len(g),
            'min_year_available': years.min() if len(years) else None,
            'max_year_available': years.max() if len(years) else None,
            #'reference_year': reference_year,
            'n_expected_years': len(expected_range),
            'n_missing_years': len(missing_years),
            'completeness_pct_years': round(100 * len(actual_years) / len(expected_range), 2) if expected_range else None,
            'longest_gap_years': longest_gap,

            'min_year_nonnull': min_year_nonnull,
            'max_year_nonnull': max_year_nonnull,
            'n_null_values': g[value_col].isna().sum(),
            'completeness_pct_nonnull': round(100 * len(actual_years_nonnull) / len(actual_years), 2) if actual_years else None,
            'last_5_years_available': last_5_available,
            'last_5_years_pct': round(100 * last_5_available / len(last_5_range), 2),
            'last_10_years_available': last_10_available,
            'last_10_years_pct': round(100 * last_10_available / len(last_10_range), 2),
            'consecutive_repeated_values (3+ years)': ', '.join(ranges) if ranges else None,
            'n_repeated_value_runs': len(ranges),
        })
    return pd.DataFrame(results)


In [22]:
def statistical_metrics(df, group_cols=['KeyID','Series name', 'Country', 'ISO'], year_col='Year', value_col='data'):
    results = []
    for keys, g in df.groupby(group_cols):
        g = g.sort_values(year_col)
        g_valid = g.dropna(subset=[year_col, value_col])
        years = g_valid[year_col].astype(int).values
        vals = g_valid[value_col].values
        n_obs = len(vals)
        
        row = {**dict(zip(group_cols, keys)), 'n_nonnull_obs': n_obs}
        
        # hard floor -- can't compute anything meaningful below 2 points
        if n_obs < 2:
            row['data_confidence'] = 'insufficient'
            results.append(row)
            continue
        
        # tiered confidence based on sample size
        if n_obs < 5:
            row['data_confidence'] = 'very_low'
        elif n_obs < 8:
            row['data_confidence'] = 'low'
        elif n_obs < 20:
            row['data_confidence'] = 'moderate'
        else:
            row['data_confidence'] = 'high'
        
        # --- gap-aware year-over-year change ---
        year_gaps = np.diff(years)
        val_series = pd.Series(vals)
        raw_pct_change = val_series.pct_change() * 100
        
        # flag inf/-inf from 0 -> nonzero transitions before they poison max/mean
        n_inf_changes = np.isinf(raw_pct_change).sum()
        raw_pct_change_clean = raw_pct_change.replace([np.inf, -np.inf], np.nan)
        
        is_adjacent_year = np.concatenate([[False], year_gaps == 1])
        true_yoy = raw_pct_change_clean[is_adjacent_year]
        gap_change = raw_pct_change_clean[~is_adjacent_year & raw_pct_change_clean.notna()]
        
        # --- density ---
        span_years = years.max() - years.min() + 1
        density_pct = round(100 * n_obs / span_years, 2)
        
        vals_s = pd.Series(vals)
        q1, q3 = vals_s.quantile(0.25), vals_s.quantile(0.75)
        iqr = q3 - q1
        outlier_mask = (vals_s < q1 - 1.5 * iqr) | (vals_s > q3 + 1.5 * iqr)
        
        z_scores = (vals_s - vals_s.mean()) / vals_s.std() if vals_s.std() != 0 else vals_s * 0
        z_outliers = (z_scores.abs() > 3).sum()
        
        direction = np.sign(val_series.diff().dropna())
        sign_flips = (direction != direction.shift()).sum() - 1 if len(direction) > 1 else 0
        
        row.update({
            'n_expected_span_years': span_years,
            'data_density_pct': density_pct,
            'iqr_reliable': n_obs >= 8,
            'zscore_reliable': n_obs >= 20,
            'mean': vals_s.mean(),
            'std': vals_s.std(),
            'cv_pct': round(100 * vals_s.std() / vals_s.mean(), 2) if vals_s.mean() != 0 else None,
            'n_inf_changes': int(n_inf_changes),
            'max_true_yoy_change_pct': true_yoy.abs().max() if len(true_yoy) else None,
            'mean_true_yoy_change_pct': true_yoy.abs().mean() if len(true_yoy) else None,
            'n_true_yoy_obs': len(true_yoy),
            'n_large_true_jumps_gt50pct': (true_yoy.abs() > 50).sum() if len(true_yoy) else 0,
            'max_gap_change_pct': gap_change.abs().max() if len(gap_change) else None,
            'n_gap_changes': len(gap_change),
            'n_iqr_outliers': outlier_mask.sum(),
            'n_zscore_outliers': z_outliers,
            'n_sign_flips': max(sign_flips, 0),
            'n_negative_values': (vals_s < 0).sum(),
            'n_zero_values': (vals_s == 0).sum(),
        })
        results.append(row)
    return pd.DataFrame(results)



In [23]:
structural = structural_metrics(index_data_long)
statistical = statistical_metrics(index_data_long)


In [24]:

output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / 'Data Quality Metrics - Index 9.xlsx'

sheets = {
    'Structural Metrics' : structural,
    'Statistical Metrics' : statistical,
    'Data Long Format' : index_data_long
}

write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
    )

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\Data Quality Metrics - Index 9.xlsx
